# 01 — Sanity Check: GPT-4o tokenizer on Korean

**Goal**: confirm the toolchain (tiktoken + our `OpenAITokenizer` wrapper) produces token counts in the expected range for Korean text *before* we wire up more providers and corpora.

**Pass criterion**

| Language | Expected TPC (tokens / character) |
|---|---|
| Korean | ~0.5 – 0.8 |
| English | ~0.20 – 0.30 |

If Korean lands well outside `0.5–0.8`, something is wrong (encoding, normalization, model name) — fix that before trusting any later experiment.

**Phase 1 surprise hypotheses to keep in mind** (we are not testing them here yet, but later runs will):

- **H1** — frontier models meaningfully improved Korean efficiency  
- **H2** — Korean-specialized models win on TPC but lose on ECPC after pricing  
- **H3** — medical / 한자-mixed text has a far higher penalty than everyday text

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Make the in-tree package importable without installing it.
# Once you `uv sync` and `pip install -e .` you can drop these two lines.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from korean_llm_cost.tokenizers.openai_tok import OpenAITokenizer

tok = OpenAITokenizer("gpt-4o")
print(tok)

In [ ]:
# 10 Korean samples across categories we plan to measure in Phase 1,
# plus a few harder cases (한자 mixed, code mixed, exclamation) to see
# the per-sample variance even at this tiny scale.
korean_samples: list[tuple[str, str]] = [
    ("conv",       "오늘 점심 뭐 먹을까?"),
    ("conv",       "내일 회의 시간 좀 알려줘."),
    ("news",       "한국은행은 기준금리를 동결하기로 결정했다."),
    ("news",       "정부는 내년도 예산안을 국회에 제출하였다."),
    ("medical",    "환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다."),
    ("medical",    "고혈압성 좌심실 비대 소견이 관찰되었다."),
    ("academic",   "본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다."),
    ("code_mixed", "리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다."),
    ("hanja",      "大韓民國의 헌법 제1조는 민주공화국임을 천명한다."),
    ("short",      "와 진짜 대박이네요!"),
]

english_baseline: list[str] = [
    "What should we eat for lunch today?",
    "Let me know the meeting time tomorrow.",
    "The Bank of Korea decided to freeze the benchmark interest rate.",
    "This study quantitatively analyzes the efficiency of Korean tokenizers.",
    "Wow, that's really amazing!",
]

len(korean_samples), len(english_baseline)

In [ ]:
def tpc(text: str) -> float:
    """Tokens per character. Aggregate over many texts via summed counts,
    not averaged TPCs — averaging per-text TPC over-weights short texts."""
    return tok.count(text) / len(text) if text else 0.0


print(f"{'category':<12} {'chars':>5} {'toks':>5} {'tpc':>6}  text")
print("-" * 80)
for category, text in korean_samples:
    n_chars = len(text)
    n_toks = tok.count(text)
    print(f"{category:<12} {n_chars:>5} {n_toks:>5} {n_toks/n_chars:>6.3f}  {text}")

In [ ]:
# Aggregate TPC: total tokens / total chars (NOT mean of per-text TPCs).
ko_chars = sum(len(t) for _, t in korean_samples)
ko_toks = sum(tok.count(t) for _, t in korean_samples)
en_chars = sum(len(t) for t in english_baseline)
en_toks = sum(tok.count(t) for t in english_baseline)

ko_tpc = ko_toks / ko_chars
en_tpc = en_toks / en_chars
kpr = ko_tpc / en_tpc  # rough; real KPR needs aligned parallel data

print(f"Korean : {ko_toks:>4} tokens / {ko_chars:>4} chars = TPC {ko_tpc:.3f}")
print(f"English: {en_toks:>4} tokens / {en_chars:>4} chars = TPC {en_tpc:.3f}")
print(f"Rough KPR (ko / en) = {kpr:.2f}x")

# Pass / fail messages — make it impossible to miss a regression.
ok_ko = 0.5 <= ko_tpc <= 0.8
ok_en = 0.20 <= en_tpc <= 0.30
print()
print("Korean TPC in [0.5, 0.8]?  ", "PASS" if ok_ko else f"FAIL ({ko_tpc:.3f})")
print("English TPC in [0.20, 0.30]?", "PASS" if ok_en else f"FAIL ({en_tpc:.3f})")

## What to do if a check fails

- **Korean TPC > 0.8**: likely encoding-related — confirm the source file is UTF-8, no NFC/NFD inconsistency, and `tok.count` is being called on the raw `str` (not bytes).
- **Korean TPC < 0.5**: implausible for Hangul on `o200k_base`. Verify `tok.info.version == '2024-11-20'` (i.e., the `gpt-4o` snapshot, not a fallback).
- **English TPC > 0.30**: check whether punctuation-heavy or capitalized samples are dominating; rule out trailing whitespace or BOM characters.

## Next

Once both checks pass, we have a green light to (a) wire up Anthropic + Google + HF tokenizers behind the same `Tokenizer` interface and (b) start pulling actual corpora into `data/corpora/`. We'll come back to this notebook only if a later result looks suspicious — it's the regression baseline.